# 07 Problem 2b: new fakes with SDXL-Turbo
T4 GPU, about 10 min. 500 fakes from a generator released after GenImage, used as a new unseen test set in nb 08.

Prompts follow GenImage ("a photo of a {ImageNet class}"), one class per image, one seed per image. Everything goes through the SAME preprocessing (crop, 224, JPEG q96) so there is no format shortcut.

Refs: SDXL-Turbo / ADD (Sauer et al. 2023, arXiv 2311.17042), diffusers (github.com/huggingface/diffusers), GenImage (arXiv 2306.08571)

In [ ]:
# setup (same in every notebook)
import os, sys, subprocess

try:  # autoreload breaks on some Colab Python versions
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    print("autoreload unavailable, restart runtime after pulling changes")

REPO_URL = "https://github.com/Devanshu1503/CV_project_genimage-detection.git"
REPO_DIR = "/content/CV_project_genimage-detection"

if "google.colab" in sys.modules:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], capture_output=True, text=True)
        print("[setup] git pull:", (r.stdout or r.stderr).strip())
else:  # local run from notebooks/
    REPO_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from src.colab_setup import setup
cfg, P = setup()  # config dict + paths

## 1. GPU check

In [ ]:
import torch

ALLOW_CPU = False   # CPU is way too slow, debug only
if torch.cuda.is_available():
    DEVICE = "cuda"
    print("GPU OK:", torch.cuda.get_device_name(0))
else:
    DEVICE = "cpu"
    msg = ("NO GPU FOUND. In Colab: Runtime -> Change runtime type -> T4 GPU, then Runtime -> Restart "
           "and run all again.")
    if not ALLOW_CPU:
        raise RuntimeError(msg)
    print("WARNING:", msg)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image

from src import datasets as ds
from src import eda_utils as eu
from src import generate as gen
from src import image_utils as iu
from src.config import generator_names, get_logger

eu.set_style()
log = get_logger("07_sdxl_turbo_generation", P.logs)
SEED = cfg["project"]["seed"]
NAMES = generator_names(cfg)
PP = cfg["preprocess"]
SX = cfg["problem2"]["sdxl"]
OUT = P.project_root / "problem2" / SX["folder"]   # our own folder, nothing shared is touched
RAW, PROC = OUT / "raw", OUT / "processed"
PROC_REL = f"problem2/{SX['folder']}/processed"   # relative to genimage_project, like processed_relpath
FIG = P.figures / "problem2"


def save(fig, name):
    eu.savefig(fig, f"07_{name}", FIG)
    plt.show()

## 2. Prompt plan
Saved once; a rerun checks it is identical (same classes, same seeds).

In [ ]:
plan = gen.make_prompt_plan(SX["n_images"], SEED, SX["prompt_template"])
plan_file = OUT / "prompts.csv"
if plan_file.exists():
    old = pd.read_csv(plan_file)
    assert old[["prompt", "seed", "filename"]].equals(plan[["prompt", "seed", "filename"]]), "prompt plan changed"
    print("prompts.csv exists and matches")
else:
    OUT.mkdir(parents=True, exist_ok=True)
    plan.to_csv(plan_file, index=False)
    print(f"saved {plan_file}")
display(plan.head())

## 3. Generate
fp16, 1 step, no guidance (how SDXL-Turbo is meant to be used). Resumable: existing PNGs are skipped.

In [ ]:
meta = gen.generation_metadata(SX["model_id"], SX["steps"], SX["guidance"], SX["size"], SX["prompt_template"], SEED)
gen.save_json(meta, OUT / "generation_meta.json")
display(pd.Series(meta))

todo = sum(not (RAW / f).exists() for f in plan["filename"])
print(f"{todo} images to generate")
if todo:
    pipe = gen.load_pipeline(SX["model_id"], DEVICE, fp16=DEVICE == "cuda")
    gen.generate_from_plan(pipe, plan, RAW, SX["steps"], SX["guidance"], SX["size"], SX["batch_size"], DEVICE)
    del pipe

### Check the outputs
fp16 VAE problems show up as black / flat images.

In [ ]:
rows = []
for f in plan["filename"]:
    a = np.asarray(Image.open(RAW / f).convert("RGB"), dtype=np.float32)
    rows.append({"filename": f, "w": a.shape[1], "h": a.shape[0], "mean": a.mean(), "std": a.std()})
chk = pd.DataFrame(rows)
bad = chk.loc[(chk["std"] < 2) | (chk["mean"] < 2)]
print(f"{len(chk)} images, sizes: {chk.groupby(['w', 'h']).size().to_dict()}, flat/black: {len(bad)}")
if len(bad):
    display(bad)  # delete these files and rerun the generate cell with other seeds if this happens

In [ ]:
ex = plan.sample(16, random_state=SEED)
fig = eu.plot_image_grid([RAW / f for f in ex["filename"]], ex["class_name"].str.slice(0, 22).tolist(), ncols=8,
                         title="SDXL-Turbo samples (raw 512 px)")
save(fig, "sdxl_turbo_samples")

## 4. Same preprocessing as everything else
`image_utils.process_file`: RGB, center crop, 224, JPEG q96.

In [ ]:
pairs = [(RAW / f, PROC / f.replace(".png", ".jpg")) for f in plan["filename"]]
errors = iu.process_many(pairs, n_threads=PP["n_threads"], size=PP["image_size"], quality=PP["jpeg_quality"],
                         subsampling=PP["jpeg_subsampling"], resample=PP["resample"])
print("processing errors:", sum(e is not None for e in errors))

index = ds.load_index(P.manifests)
real_ref = index.loc[(index["split_set"] == "test") & (index["generator"] == SX["real_source"]) & (index["y"] == 0)]
meta_new = pd.DataFrame(iu.metadata_many([d for _, d in pairs], n_threads=PP["n_threads"], with_hashes=False))
meta_real = pd.DataFrame(iu.metadata_many([P.project_root / r for r in real_ref["processed_relpath"]],
                                          n_threads=PP["n_threads"], with_hashes=False))
cmp = pd.concat([meta_new.assign(source="SDXL-Turbo fake"), meta_real.assign(source="real")])
display(cmp.groupby("source")[["real_format", "width", "height", "jpeg_quality"]].agg(lambda s: s.value_counts().to_dict()))

## 5. Test index for the new generator
The 500 SDXL-Turbo fakes + the 500 real test images of the SD1.4 test set (already unseen by every model).

In [ ]:
sdxl_index = gen.build_sdxl_index(plan, PROC_REL, real_ref, SX["folder"], SX["name"])
assert all((P.project_root / r).exists() for r in sdxl_index["processed_relpath"])
sdxl_index.to_csv(OUT / "sdxl_index.csv", index=False)
print(f"saved {OUT / 'sdxl_index.csv'}")
display(sdxl_index.groupby("label").size())

## 6. Frequency check (after preprocessing)
Same FFT view as nb 02. Does SDXL-Turbo carry the SD-style high-frequency excess?

In [ ]:
E = cfg["eda"]
sd_fakes = index.loc[(index["split_set"] == "test") & (index["generator"] == SX["real_source"]) & (index["y"] == 1)]
fft_df = pd.concat([real_ref.assign(source="Real"), sd_fakes.assign(source=NAMES[SX["real_source"]]),
                    sdxl_index.loc[sdxl_index["y"] == 1].assign(source=SX["name"])])
fft_df = fft_df.groupby("source").sample(n=min(E["fft_per_group"], fft_df.groupby("source").size().min()), random_state=SEED)
spectra, _ = eu.average_spectra(fft_df, P.project_root, "processed_relpath", "source", E["fft_crop"])
spectra = {k: spectra[k] for k in ["Real", NAMES[SX["real_source"]], SX["name"]] if k in spectra}
fig = eu.plot_spectra(spectra, reference="Real", ncols=3, title="Mean log-|FFT| after preprocessing")
save(fig, "fft_sdxl_turbo")

## Findings
Fill in after the run.
- Generated: ... images, ... flat/black, model revision (generation_meta.json): ...
- After preprocessing: format/size/quality identical to the reals? (table in section 4)
- FFT: does SDXL-Turbo look like SD1.4 (latent diffusion, same VAE family) or different?

## Run before closing the tab
Uploads pending Drive writes, otherwise the next session may not see them.

In [ ]:
from src.colab_setup import flush_drive
flush_drive()